# LiteEdit-Qwen — 03: LoRA Fine-tuning

**Goal:** Train a rank-8 LoRA adapter and measure quality recovery vs INT4 baseline.

**Key question:** Does a ~20 MB adapter recover LPIPS lost to quantization?

**Outputs:**
- `results/lora_checkpoints/bg_replace_adapter/` — trained weights
- `results/figures/lora_comparison.png` — visual side-by-side
- `results/metrics/lora_impact.csv` — INT4 vs INT4+LoRA numbers

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from paths import PATHS
print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Adapters will be saved to: {PATHS.lora_checkpoints}')

In [ ]:
# ── Build training data if needed ────────────────────────────────
if not (PATHS.results.parent / 'data' / 'train' / 'bg_replace').exists():
    !python data/dataset_bg.py --build_demo --data_dir data --n_images 30
    print('Training data created.')
else:
    print('Training data already present.')

In [ ]:
# ── Train bg_replace adapter ──────────────────────────────────────
# output_dir defaults to results/lora_checkpoints/bg_replace_adapter
!python train/train_lora.py \
    --task bg_replace \
    --data_dir data/train/bg_replace \
    --base_model Qwen/Qwen2.5-VL-7B-Instruct \
    --quantize \
    --rank 8 \
    --epochs 1 \
    --batch_size 1

In [ ]:
# ── Inspect saved adapter ─────────────────────────────────────────
adapter_dir = PATHS.lora_adapter_dir('bg_replace')
print(f'Adapter directory: {adapter_dir}')

# Print file sizes
total_mb = 0
for f in sorted(adapter_dir.iterdir()):
    mb = f.stat().st_size / 1e6
    total_mb += mb
    print(f'  {f.name:<40} {mb:.1f} MB')
print(f'  {"Total":40} {total_mb:.1f} MB')

# Print training metadata
meta_path = adapter_dir / 'train_meta.json'
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print('\nTraining metadata:')
    for k, v in meta.items():
        print(f'  {k}: {v}')

In [ ]:
# ── Compare: INT4 baseline vs INT4+LoRA ───────────────────────────
import time, torch, gc
from models.loader import load_model
from tasks.bg_replace import BgReplaceTask
from eval.metrics import MetricsTracker

test_img = Image.open('data/test/bg_replace/input/img_0000.png').convert('RGB')
ref_img  = Image.open('data/test/bg_replace/ref/img_0000.png').convert('RGB')
prompt   = 'Replace the background with a snowy mountain landscape'
tracker  = MetricsTracker(device='cuda')

results = []
for label, cfg_path in [('INT4 baseline', 'configs/quant_int4.yaml'),
                         ('INT4 + LoRA',   'configs/lora_bg.yaml')]:
    model, proc, cfg = load_model(cfg_path)
    task = BgReplaceTask(model, proc, cfg)

    tracker.start_timer()
    out, _ = task.run(image=test_img, prompt=prompt)
    lat, vram = tracker.stop_timer()

    m = tracker.compute(pred=out, ref=ref_img)
    results.append({'label': label, 'output': out,
                    'lpips': m['lpips'], 'psnr': m['psnr'],
                    'latency_s': round(lat,3), 'peak_vram_gb': round(vram,3)})

    del model, task; torch.cuda.empty_cache(); gc.collect()

print('\n═══ LoRA Impact ═══')
print(f'{"Config":<20} {"LPIPS↓":>8} {"PSNR↑":>8} {"Lat(s)":>8} {"VRAM":>7}')
print('─' * 55)
for r in results:
    print(f'{r["label"]:<20} {r["lpips"]:>8.4f} {r["psnr"]:>8.2f} '
          f'{r["latency_s"]:>8.2f} {r["peak_vram_gb"]:>6.2f}G')
print('═' * 55)

In [ ]:
# ── Save figure → results/figures/ ───────────────────────────────
PATHS.figures.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, len(results)+1, figsize=(5*(len(results)+1), 4))
axes[0].imshow(test_img); axes[0].set_title('Input'); axes[0].axis('off')

for i, r in enumerate(results):
    axes[i+1].imshow(r['output'])
    axes[i+1].set_title(f"{r['label']}\nLPIPS={r['lpips']:.4f}  PSNR={r['psnr']:.1f}dB")
    axes[i+1].axis('off')

plt.suptitle('bg_replace: INT4 baseline vs INT4 + LoRA', y=1.02)
plt.tight_layout()

fig_path = PATHS.figures / 'lora_comparison.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')

In [ ]:
# ── Save metrics CSV → results/metrics/ ──────────────────────────
df = pd.DataFrame([{k: v for k, v in r.items() if k != 'output'} for r in results])

csv_path = PATHS.metrics / 'lora_impact.csv'
df.to_csv(csv_path, index=False)
print(f'Metrics saved → {csv_path}')
df

## Interpreting results

| Result | Interpretation |
|---|---|
| LPIPS decreases with LoRA | Adapter successfully recovers perceptual quality |
| LPIPS unchanged | Adapter underfitted — try more epochs or higher rank |
| Latency increase < 5% | Expected — adapter is tiny vs base model |
| VRAM increase < 0.2 GB | Expected for rank-8 |

**After training all three adapters**, run `bash scripts/run_ablation.sh` to get
the complete ablation table with all configs × all tasks.

All outputs are in `results/`:
- `results/lora_checkpoints/` — adapter weights
- `results/figures/lora_comparison.png` — visual
- `results/metrics/lora_impact.csv` — numbers